# Joint Gini sweep analyzer

Compare the same Gini coefficients in abundant and single-shot, using learning seed 0.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 320)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root.")


REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"

SANITY_DIR = RESULTS_ROOT / "final_gini_sanity_analysis_2026_09_28"
SS_DIR = RESULTS_ROOT / "experiment_2026_09_29_single_shot_gini_joint_sweep"
AB_DIR = RESULTS_ROOT / "experiment_2026_09_29_abundant_gini_joint_sweep"

AN_DIR = RESULTS_ROOT / "joint_gini_sweep_analysis_2026_09_29"
AN_DIR.mkdir(parents=True, exist_ok=True)

anchor_path = SANITY_DIR / "gini_vs_unregularized_seed_level.csv"
ss_path = SS_DIR / "single_shot_gini_sweep_seed0.csv"
ab_path = AB_DIR / "abundant_gini_joint_sweep_seed0.csv"

for p in [anchor_path, ss_path, ab_path]:
    if not p.exists():
        raise FileNotFoundError(p)

print("Repo:", REPO_ROOT)


Repo: d:\Work\repos\RL\unknown_graph_networks


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_85800\2198010353.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
METRICS = [
    "control_final",
    "prediction_ratio",
    "A_MAE",
    "A_Fro",
    "row_TV",
    "true_edge_mass",
    "support_topdegree_recall",
    "edge_AP",
    "centrality_L1",
]

KEYS = [
    "regime",
    "environment_id",
    "scenario_class",
    "family",
    "dynamics",
    "topology_seed",
    "opinion_seed",
    "environment_seed",
    "learning_seed",
]

anchor = pd.read_csv(anchor_path)
anchor = anchor[anchor["learning_seed"] == 0].copy()
assert len(anchor) == 390

anchor_rows = []
for lam, suffix in [(0.0, "old"), (1e-4, "new")]:
    block = anchor[KEYS].copy()
    block["regularizer_lambda"] = lam
    for metric in METRICS:
        block[metric] = anchor[f"{metric}_{suffix}"].to_numpy()
    anchor_rows.append(block)

anchors = pd.concat(anchor_rows, ignore_index=True)

ss = pd.read_csv(ss_path)
ss_can = ss[
    [
        "environment_id", "scenario_class", "family", "dynamics",
        "topology_seed", "opinion_seed", "environment_seed",
        "learning_seed", "regularizer_lambda",
        "final_mean", "final_fit_model_over_identity",
        "A_MAE", "A_Fro", "row_TV", "true_edge_mass",
        "support_topdegree_recall", "edge_AP", "centrality_L1",
    ]
].copy()
ss_can.insert(0, "regime", "single_shot")
ss_can = ss_can.rename(
    columns={
        "final_mean": "control_final",
        "final_fit_model_over_identity": "prediction_ratio",
    }
)

ab = pd.read_csv(ab_path)
ab_can = ab[
    [
        "environment_id", "scenario_class", "family", "dynamics",
        "topology_seed", "opinion_seed", "environment_seed",
        "learning_seed", "regularizer_lambda",
        "mean_end", "model_over_identity",
        "A_MAE", "A_Fro", "row_TV", "true_edge_mass",
        "support_topdegree_recall", "edge_AP", "centrality_L1",
    ]
].copy()
ab_can.insert(0, "regime", "abundant")
ab_can = ab_can.rename(
    columns={
        "mean_end": "control_final",
        "model_over_identity": "prediction_ratio",
    }
)

work = pd.concat([anchors, ss_can, ab_can], ignore_index=True, sort=False)

expected_lambdas = {0.0, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4}
assert set(work["regularizer_lambda"].astype(float).unique()) == expected_lambdas
assert work.groupby(["regime", "regularizer_lambda"]).size().eq(195).all()

print("Conditions:", len(expected_lambdas), "per regime")
display(
    work.groupby(["regime", "regularizer_lambda"])
    .size().rename("n").reset_index()
)


Conditions: 6 per regime


,regime,regularizer_lambda,n
0,abundant,0.000000,195
1,abundant,0.000001,195
2,abundant,0.000003,195
3,abundant,0.000010,195
4,abundant,0.000030,195
5,abundant,0.000100,195
6,single_shot,0.000000,195
7,single_shot,0.000001,195
8,single_shot,0.000003,195
9,single_shot,0.000010,195


In [3]:
ref = (
    work[work["regularizer_lambda"] == 0.0]
    .set_index(["regime", "environment_id"])
)

for metric in METRICS:
    work[f"delta_{metric}"] = work.apply(
        lambda r: float(r[metric]) - float(
            ref.loc[(r["regime"], r["environment_id"]), metric]
        ),
        axis=1,
    )

summary = (
    work.groupby(["regime", "regularizer_lambda"], as_index=False)
    .agg(
        n=("environment_id", "size"),
        control=("control_final", "mean"),
        delta_control=("delta_control_final", "mean"),
        prediction=("prediction_ratio", "mean"),
        delta_prediction=("delta_prediction_ratio", "mean"),
        row_TV=("row_TV", "mean"),
        delta_row_TV=("delta_row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        delta_true_edge_mass=("delta_true_edge_mass", "mean"),
        edge_AP=("edge_AP", "mean"),
        delta_edge_AP=("delta_edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        delta_centrality_L1=("delta_centrality_L1", "mean"),
    )
)

by_dyn = (
    work.groupby(
        ["regime", "regularizer_lambda", "scenario_class", "dynamics"],
        as_index=False,
    )
    .agg(
        n=("environment_id", "size"),
        control=("control_final", "mean"),
        delta_control=("delta_control_final", "mean"),
        prediction=("prediction_ratio", "mean"),
        delta_prediction=("delta_prediction_ratio", "mean"),
        row_TV=("row_TV", "mean"),
        delta_row_TV=("delta_row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        delta_true_edge_mass=("delta_true_edge_mass", "mean"),
        edge_AP=("edge_AP", "mean"),
        delta_edge_AP=("delta_edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        delta_centrality_L1=("delta_centrality_L1", "mean"),
    )
)

worst = (
    by_dyn.groupby(["regime", "regularizer_lambda"], as_index=False)
    .agg(
        worst_dynamic_control_delta=("delta_control", "min"),
        worst_dynamic_prediction_delta=("delta_prediction", "max"),
        worst_dynamic_row_TV_delta=("delta_row_TV", "max"),
    )
)

summary = summary.merge(
    worst,
    on=["regime", "regularizer_lambda"],
    validate="one_to_one",
)

summary.to_csv(AN_DIR / "joint_gini_summary.csv", index=False)
by_dyn.to_csv(AN_DIR / "joint_gini_by_dynamics.csv", index=False)

print("OVERALL")
display(summary.round(6))


OVERALL


,regime,regularizer_lambda,n,control,delta_control,prediction,delta_prediction,row_TV,delta_row_TV,true_edge_mass,delta_true_edge_mass,edge_AP,delta_edge_AP,centrality_L1,delta_centrality_L1,worst_dynamic_control_delta,worst_dynamic_prediction_delta,worst_dynamic_row_TV_delta
0,abundant,0.000000,195,0.768628,0.000000,0.638310,0.000000,0.575996,0.000000,0.433523,0.000000,0.741848,0.000000,0.480081,0.000000,0.000000,0.000000,0.000000
1,abundant,0.000001,195,0.768670,0.000042,0.638264,-0.000047,0.575897,-0.000099,0.433634,0.000111,0.741801,-0.000046,0.479972,-0.000110,0.000000,-0.000000,0.000008
2,abundant,0.000003,195,0.768666,0.000038,0.638171,-0.000140,0.575699,-0.000297,0.433856,0.000332,0.741661,-0.000186,0.479752,-0.000329,-0.000000,-0.000001,0.000025
3,abundant,0.000010,195,0.768369,-0.000259,0.637848,-0.000463,0.575011,-0.000985,0.434631,0.001108,0.741016,-0.000832,0.478988,-0.001094,-0.003006,-0.000002,0.000093
4,abundant,0.000030,195,0.768263,-0.000365,0.636948,-0.001362,0.573100,-0.002896,0.436841,0.003318,0.738900,-0.002947,0.476815,-0.003266,-0.005351,-0.000002,0.000326
5,abundant,0.000100,195,0.767630,-0.000998,0.634230,-0.004080,0.567461,-0.008535,0.444259,0.010735,0.728919,-0.012928,0.469769,-0.010313,-0.008300,0.000074,0.001730
6,single_shot,0.000000,195,0.763121,0.000000,0.171743,0.000000,0.383458,0.000000,0.656915,0.000000,0.704522,0.000000,0.502002,0.000000,0.000000,0.000000,0.000000
7,single_shot,0.000001,195,0.762683,-0.000437,0.178234,0.006491,0.393633,0.010175,0.661612,0.004698,0.694185,-0.010338,0.536644,0.034642,-0.004419,0.040329,0.063477
8,single_shot,0.000003,195,0.760911,-0.002209,0.200492,0.028748,0.412045,0.028587,0.672133,0.015218,0.671978,-0.032544,0.603604,0.101602,-0.013244,0.124539,0.109603
9,single_shot,0.000010,195,0.755084,-0.008036,0.259979,0.088235,0.453546,0.070088,0.672588,0.015673,0.622500,-0.082023,0.741855,0.239852,-0.025861,0.282586,0.236152


In [4]:
view_metrics = [
    "delta_control",
    "delta_prediction",
    "delta_row_TV",
    "delta_true_edge_mass",
    "delta_edge_AP",
    "delta_centrality_L1",
    "worst_dynamic_control_delta",
    "worst_dynamic_prediction_delta",
    "worst_dynamic_row_TV_delta",
]

wide_parts = []
for regime in ["abundant", "single_shot"]:
    part = summary[summary["regime"] == regime][
        ["regularizer_lambda"] + view_metrics
    ].copy()
    part = part.rename(
        columns={
            c: f"{regime}_{c}"
            for c in view_metrics
        }
    )
    wide_parts.append(part)

joint = wide_parts[0].merge(
    wide_parts[1],
    on="regularizer_lambda",
    validate="one_to_one",
).sort_values("regularizer_lambda")

joint.to_csv(AN_DIR / "joint_gini_side_by_side.csv", index=False)

print("SIDE BY SIDE — deltas vs no regularization")
display(joint.round(6))


SIDE BY SIDE — deltas vs no regularization


,regularizer_lambda,abundant_delta_control,abundant_delta_prediction,abundant_delta_row_TV,abundant_delta_true_edge_mass,abundant_delta_edge_AP,abundant_delta_centrality_L1,abundant_worst_dynamic_control_delta,abundant_worst_dynamic_prediction_delta,abundant_worst_dynamic_row_TV_delta,single_shot_delta_control,single_shot_delta_prediction,single_shot_delta_row_TV,single_shot_delta_true_edge_mass,single_shot_delta_edge_AP,single_shot_delta_centrality_L1,single_shot_worst_dynamic_control_delta,single_shot_worst_dynamic_prediction_delta,single_shot_worst_dynamic_row_TV_delta
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000001,0.000042,-0.000047,-0.000099,0.000111,-0.000046,-0.000110,0.000000,-0.000000,0.000008,-0.000437,0.006491,0.010175,0.004698,-0.010338,0.034642,-0.004419,0.040329,0.063477
2,0.000003,0.000038,-0.000140,-0.000297,0.000332,-0.000186,-0.000329,-0.000000,-0.000001,0.000025,-0.002209,0.028748,0.028587,0.015218,-0.032544,0.101602,-0.013244,0.124539,0.109603
3,0.000010,-0.000259,-0.000463,-0.000985,0.001108,-0.000832,-0.001094,-0.003006,-0.000002,0.000093,-0.008036,0.088235,0.070088,0.015673,-0.082023,0.239852,-0.025861,0.282586,0.236152
4,0.000030,-0.000365,-0.001362,-0.002896,0.003318,-0.002947,-0.003266,-0.005351,-0.000002,0.000326,-0.023257,0.171664,0.143029,-0.002031,-0.148413,0.417964,-0.044957,0.347238,0.286304
5,0.000100,-0.000998,-0.004080,-0.008535,0.010735,-0.012928,-0.010313,-0.008300,0.000074,0.001730,-0.049886,0.337772,0.265107,-0.077728,-0.230420,0.609086,-0.159192,0.744660,0.594898


In [5]:
for lam in sorted(work["regularizer_lambda"].unique()):
    if lam == 0:
        continue
    print("\n" + "=" * 100)
    print(f"lambda = {lam:g}")
    print("=" * 100)
    display(
        by_dyn[by_dyn["regularizer_lambda"] == lam][
            [
                "regime", "scenario_class", "dynamics",
                "delta_control", "delta_prediction",
                "delta_row_TV", "delta_true_edge_mass",
                "delta_edge_AP", "delta_centrality_L1",
            ]
        ].round(6)
    )

print("\nOutputs:", AN_DIR)
print("JOINT GINI SWEEP ANALYSIS COMPLETE")



lambda = 1e-06


,regime,scenario_class,dynamics,delta_control,delta_prediction,delta_row_TV,delta_true_edge_mass,delta_edge_AP,delta_centrality_L1
9,abundant,structured_mechanism,friedkinjohnsen,0.000000,-0.000008,-0.000007,0.000009,-0.000002,-0.000012
10,abundant,structured_mechanism,hegselmannkrause,0.000000,-0.000000,-0.000050,0.000113,0.000201,-0.000094
11,abundant,structured_mechanism,nonlinearinfluence,0.000000,-0.000060,-0.000100,0.000114,-0.000051,-0.000184
12,abundant,unstructured_random,coca,0.000074,-0.000051,-0.000144,0.000154,-0.000317,-0.000119
13,abundant,unstructured_random,friedkinjohnsen,0.000000,-0.000004,-0.000009,0.000010,0.000000,-0.000007
14,abundant,unstructured_random,hegselmannkrause,0.000034,-0.000036,-0.000085,0.000096,0.000005,-0.000106
15,abundant,unstructured_random,laplacian,0.000070,-0.000161,-0.000321,0.000333,-0.000087,-0.000307
16,abundant,unstructured_random,nonlinearinfluence,0.000000,-0.000066,-0.000129,0.000135,-0.000051,-0.000121
17,abundant,unstructured_random,repulsion,0.000151,-0.000004,0.000008,-0.000005,-0.000001,-0.000022
63,single_shot,structured_mechanism,friedkinjohnsen,-0.000221,-0.000200,0.001607,0.004290,-0.008363,0.016077



lambda = 3e-06


,regime,scenario_class,dynamics,delta_control,delta_prediction,delta_row_TV,delta_true_edge_mass,delta_edge_AP,delta_centrality_L1
18,abundant,structured_mechanism,friedkinjohnsen,0.000000,-0.000024,-0.000022,0.000026,-0.000008,-0.000036
19,abundant,structured_mechanism,hegselmannkrause,-0.000000,-0.000001,-0.000150,0.000342,0.000011,-0.000284
20,abundant,structured_mechanism,nonlinearinfluence,0.000000,-0.000180,-0.000300,0.000342,-0.000240,-0.000551
21,abundant,unstructured_random,coca,0.000176,-0.000154,-0.000432,0.000463,-0.000851,-0.000356
22,abundant,unstructured_random,friedkinjohnsen,0.000000,-0.000013,-0.000028,0.000029,0.000006,-0.000020
23,abundant,unstructured_random,hegselmannkrause,0.000034,-0.000107,-0.000254,0.000286,-0.000030,-0.000318
24,abundant,unstructured_random,laplacian,0.000029,-0.000481,-0.000959,0.000998,-0.000239,-0.000922
25,abundant,unstructured_random,nonlinearinfluence,0.000000,-0.000198,-0.000388,0.000405,-0.000060,-0.000363
26,abundant,unstructured_random,repulsion,0.000060,-0.000013,0.000025,-0.000015,-0.000138,-0.000066
72,single_shot,structured_mechanism,friedkinjohnsen,-0.001406,-0.000430,0.009814,0.015749,-0.026333,0.056502



lambda = 1e-05


...


lambda = 3e-05


,regime,scenario_class,dynamics,delta_control,delta_prediction,delta_row_TV,delta_true_edge_mass,delta_edge_AP,delta_centrality_L1
36,abundant,structured_mechanism,friedkinjohnsen,0.000000,-0.000244,-0.000216,0.000263,-0.000205,-0.000361
37,abundant,structured_mechanism,hegselmannkrause,-0.005351,-0.000002,-0.001286,0.003645,-0.000109,-0.003086
38,abundant,structured_mechanism,nonlinearinfluence,0.000000,-0.001768,-0.002981,0.003408,-0.002434,-0.005531
39,abundant,unstructured_random,coca,0.000613,-0.001545,-0.004320,0.004718,-0.017325,-0.003359
40,abundant,unstructured_random,friedkinjohnsen,-0.000006,-0.000131,-0.000282,0.000292,-0.000013,-0.000198
41,abundant,unstructured_random,hegselmannkrause,-0.000002,-0.001051,-0.002509,0.002837,-0.001001,-0.003119
42,abundant,unstructured_random,laplacian,-0.000009,-0.004605,-0.009280,0.009811,-0.001831,-0.009151
43,abundant,unstructured_random,nonlinearinfluence,-0.000027,-0.001953,-0.003833,0.004022,-0.000479,-0.003604
44,abundant,unstructured_random,repulsion,-0.000203,-0.000131,0.000326,-0.000188,-0.000692,-0.000659
90,single_shot,structured_mechanism,friedkinjohnsen,-0.007884,0.003783,0.065117,0.061803,-0.055958,0.206789



lambda = 0.0001


,regime,scenario_class,dynamics,delta_control,delta_prediction,delta_row_TV,delta_true_edge_mass,delta_edge_AP,delta_centrality_L1
45,abundant,structured_mechanism,friedkinjohnsen,0.000000,-0.000811,-0.000718,0.000873,-0.000878,-0.001184
46,abundant,structured_mechanism,hegselmannkrause,-0.008300,0.000074,-0.001937,0.014387,-0.004101,-0.012821
47,abundant,structured_mechanism,nonlinearinfluence,0.000000,-0.005604,-0.009602,0.011247,-0.008529,-0.018481
48,abundant,unstructured_random,coca,0.000203,-0.004559,-0.012634,0.015235,-0.071938,-0.007848
49,abundant,unstructured_random,friedkinjohnsen,-0.000182,-0.000434,-0.000937,0.000972,-0.000316,-0.000658
50,abundant,unstructured_random,hegselmannkrause,-0.000079,-0.003346,-0.007996,0.009204,-0.004308,-0.010097
51,abundant,unstructured_random,laplacian,-0.000036,-0.012976,-0.027109,0.030359,-0.013914,-0.028614
52,abundant,unstructured_random,nonlinearinfluence,0.000047,-0.006231,-0.012272,0.013134,-0.002494,-0.011794
53,abundant,unstructured_random,repulsion,-0.002756,-0.000474,0.001730,-0.001074,0.000237,-0.001938
99,single_shot,structured_mechanism,friedkinjohnsen,-0.027074,0.039474,0.200157,0.078137,-0.102870,0.436819



Outputs: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\joint_gini_sweep_analysis_2026_09_29
JOINT GINI SWEEP ANALYSIS COMPLETE
